In [ ]:
###868 코드 스테이지1 pca 52차원 +스테이지2 pca51
# ============================================
# Multiclass Ensemble (LGBM + XGB + CatBoost)
#  - Stage 1: Base Multi-EDA + 3모델 앙상블(CV-fold 평균)
#  - Stage 2: Hard-class 전용 PCA+Multi-EDA + 3모델 앙상블(CV-fold 평균)
#  - Gating: gap기준 & 하드클래스 기준 분리 집계 + CSV 저장
#  - 내부 레이블 매핑(0..K-1) 일관화
# ============================================

import os, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.linalg import inv, slogdet
#/content/drive/MyDrive/test_abnormal.csv
#/content/drive/MyDrive/sample_submission_abnormal.csv

In [ ]:
# -------- 기본 설정(여기만 손보면 됨) --------
SEED          = 45
NFOLDS        = 5
USE_GPU       = False      # GPU 없으면 False 권장
GAP_THR       = 0.00       # gap(threshold) for gating
DATA_DIR      = "/content/drive/MyDrive/"
TRAIN_FILE    = os.path.join(DATA_DIR, "train_abnormal.csv")
TEST_FILE     = os.path.join(DATA_DIR, "test_abnormal.csv")
SUB_FILE      = os.path.join(DATA_DIR, "sample_submission_abnormal.csv")
OUT_SUBMIT    = os.path.join(DATA_DIR, "final_submit_stage1_enhanced_ens3_gate0.csv")
OUT_GATINGCSV = os.path.join(DATA_DIR, "gating_flags.csv")

# 하드 클래스(2단계 대상; 원본 라벨 값 그대로)
target_classes = [0, 3, 9, 15, 16, 19]
HARD_SET       = set(target_classes)
np.random.seed(SEED)

In [ ]:
# ============================================
# 0) 데이터 로드
# ============================================
train = pd.read_csv(TRAIN_FILE)
test  = pd.read_csv(TEST_FILE)
submission = pd.read_csv(SUB_FILE)

assert {'ID','target'}.issubset(train.columns), "train에 ID/target이 필요합니다."
assert 'ID' in test.columns, "test에 ID가 필요합니다."

X_all   = train.drop(columns=['ID','target']).values
y_all   = train['target'].values
X_test  = test.drop(columns=['ID']).values
all_classes = np.unique(y_all)

In [ ]:
# ============================================
# 1) 공통 유틸 (가우시안 통계, EDA 피처)
# ============================================

def fit_gauss_by_class(X, y, classes):
    eps = 1e-5; d = X.shape[1]; I = np.eye(d)
    stats = {}
    for c in classes:
        Xc = X[y==c]
        mu = Xc.mean(axis=0)
        cov = np.cov(Xc, rowvar=False) + eps*I
        inv_cov = inv(cov)
        sign, logdet = slogdet(cov)
        stats[c] = (mu, cov, inv_cov, logdet)
    return stats

def loglik_gauss(X, mu, inv_cov, logdet):
    Xm = X - mu
    quad = np.sum(Xm @ inv_cov * Xm, axis=1)
    return -0.5*(quad + logdet)

def maha_dist(X, mu, inv_cov):
    Xm = X - mu
    return np.sqrt(np.sum(Xm @ inv_cov * Xm, axis=1))

In [ ]:
# ===== Stage-1 전용: PCA(동일차원) → LDA → 가우시안 통계 =====
def fit_base_multi_eda_s1(X_train, y_train, seed=SEED):
    classes = np.unique(y_train)

    # 1) 스케일
    scal = StandardScaler().fit(X_train)
    Xs = scal.transform(X_train)

    # 2) PCA (차원 유지)
    pca = PCA(n_components=Xs.shape[1], random_state=seed).fit(Xs)
    Xp = pca.transform(Xs)

    # 3) LDA (k-1)
    lda = LinearDiscriminantAnalysis(
        n_components=min(len(classes)-1, Xp.shape[1])
    ).fit(Xp, y_train)
    Z_lda = lda.transform(Xp)

     # 4) 클래스별 가우시안 통계 (PCA 공간)
    stats = fit_gauss_by_class(Xp, y_train, classes)
    Z_ll, Z_md = [], []
    for c in classes:
        mu, cov, invc, logdet = stats[c]
        Z_ll.append(loglik_gauss(Xp, mu, invc, logdet))
        Z_md.append(maha_dist(Xp, mu, invc))
    Z_ll = np.vstack(Z_ll).T
    Z_md = np.vstack(Z_md).T

    X_train_all = np.concatenate([Xp, Z_lda, Z_ll, Z_md], axis=1)
    return scal, pca, lda, stats, X_train_all, classes

In [ ]:
def transform_multi_eda_features_s1(X, scal, pca, lda, stats, classes):
    # 1) 스케일 → 2) PCA
    Xs = scal.transform(X)
    Xp = pca.transform(Xs)

    # 3) LDA
    Z_lda = lda.transform(Xp)

    # 4) 가우시안 통계 (PCA 공간)
    Z_ll, Z_md = [], []
    for c in classes:
        mu, cov, invc, logdet = stats[c]
        Z_ll.append(loglik_gauss(Xp, mu, invc, logdet))
        Z_md.append(maha_dist(Xp, mu, invc))
    Z_ll = np.vstack(Z_ll).T
    Z_md = np.vstack(Z_md).T

    return np.concatenate([Xp, Z_lda, Z_ll, Z_md], axis=1)

In [ ]:
# ----- Stage 2: PCA 공간에서 Multi-EDA -----
def transform_pca_multi_eda_for_infer(X, scaler, pca, lda, gstats, classes):
    Xs = scaler.transform(X)
    Xp = pca.transform(Xs)
    Z_lda = lda.transform(Xp)
    Z_ll, Z_md = [], []
    for c in classes:
        mu, cov, invc, logdet = gstats[c]
        Z_ll.append(loglik_gauss(Xp, mu, invc, logdet))
        Z_md.append(maha_dist(Xp, mu, invc))
    Z_ll = np.vstack(Z_ll).T
    Z_md = np.vstack(Z_md).T
    return np.concatenate([Xp, Z_lda, Z_ll, Z_md], axis=1)

In [ ]:
# ============================================
# 2) 3모델 학습/예측 유틸 (앙상블) — 내부 레이블 매핑 고정
# ============================================
def fit_3models_multiclass(X, y, seed=SEED, use_gpu=USE_GPU, class_order=None):
    """
    [단일모델] XGBoost만 학습.
    반환: ( (xgb,), label_to_idx, idx_to_label )
    """
    if class_order is None:
        class_order = np.unique(y)
    class_order = np.array(class_order)

    label_to_idx = {c: i for i, c in enumerate(class_order)}
    idx_to_label = list(class_order)
    y_idx = np.vectorize(label_to_idx.get)(y)
    n_class = len(class_order)

    xgb_params = dict(
        objective="multi:softprob",
        num_class=n_class,
        random_state=seed,
        n_estimators=2500,      # (기존보다 살짝 줄여 속도 확보; 필요시 3500으로)
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=1.5,
        tree_method=("gpu_hist" if use_gpu else "hist"),
        predictor=("gpu_predictor" if use_gpu else "auto"),
        n_jobs=-1
    )
    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X, y_idx)

    # 중요: 나머지 코드와의 호환을 위해 '튜플(models), label_to_idx, idx_to_label' 형태로 반환
    return (xgb,), label_to_idx, idx_to_label

In [ ]:
def predict_proba_ens3(models_pack, X, class_order):
    """
    models_pack: ((model1, [model2, model3, ...]), label_to_idx, idx_to_label)
    - 단일 XGB 또는 3모델 앙상블 모두 지원
    """
    models, label_to_idx, idx_to_label = models_pack
    
    def _align(p):
        aligned = np.zeros((X.shape[0], len(class_order)))
        for j, c in enumerate(class_order):
            aligned[:, j] = p[:, label_to_idx[c]]
        return aligned

    probs = []
    for mdl in models:
        p = mdl.predict_proba(X)   # (n, K_local=K) — 우리는 label_to_idx로 정렬 고정
        probs.append(_align(p))

    # 단일 모델이면 그대로, 여러 개면 평균
    return probs[0] if len(probs) == 1 else sum(probs) / len(probs)


In [ ]:
# ============================================
# 3) Stage 2 CV (PCA+Multi-EDA) — 3모델 앙상블
# ============================================
def run_cv_hard_pca_multi_eda(X, y, classes, n_splits=NFOLDS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros_like(y)
    scores = []
    models = []  # (scaler, pca, lda, gstats, models_ens_pack)

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        X_tr_raw, X_va_raw = X[tr], X[va]
        y_tr, y_va         = y[tr], y[va]

        scaler = StandardScaler().fit(X_tr_raw)
        X_tr_std = scaler.transform(X_tr_raw)
        X_va_std = scaler.transform(X_va_raw)
######################################여기 !!!!!!!
        pca_dim = min(51, X_tr_std.shape[1])
        pca = PCA(n_components=pca_dim, random_state=seed).fit(X_tr_std)
        X_tr_p = pca.transform(X_tr_std)
        X_va_p = pca.transform(X_va_std)
###solver="lsqr", shrinkage="auto",
        lda = LinearDiscriminantAnalysis(
          #  solver="lsqr", shrinkage="auto",
            n_components=min(len(classes)-1, X_tr_p.shape[1])
        ).fit(X_tr_p, y_tr)
        Z_tr_lda = lda.transform(X_tr_p)
        Z_va_lda = lda.transform(X_va_p)

        gstats = fit_gauss_by_class(X_tr_p, y_tr, classes)

        Z_tr_ll, Z_va_ll, Z_tr_md, Z_va_md = [], [], [], []
        for c in classes:
            mu, cov, invc, logdet = gstats[c]
            Z_tr_ll.append(loglik_gauss(X_tr_p, mu, invc, logdet))
            Z_va_ll.append(loglik_gauss(X_va_p, mu, invc, logdet))
            Z_tr_md.append(maha_dist(X_tr_p, mu, invc))
            Z_va_md.append(maha_dist(X_va_p, mu, invc))
        Z_tr_ll = np.vstack(Z_tr_ll).T
        Z_va_ll = np.vstack(Z_va_ll).T
        Z_tr_md = np.vstack(Z_tr_md).T
        Z_va_md = np.vstack(Z_va_md).T

        X_tr_all = np.concatenate([X_tr_p, Z_tr_lda, Z_tr_ll, Z_tr_md], axis=1)
        X_va_all = np.concatenate([X_va_p, Z_va_lda, Z_va_ll, Z_va_md], axis=1)

        ens_pack = fit_3models_multiclass(
            X_tr_all, y_tr, seed=seed, use_gpu=USE_GPU, class_order=classes
        )
        proba_va = predict_proba_ens3(ens_pack, X_va_all, classes)
        pred_va  = np.array(classes)[np.argmax(proba_va, axis=1)]

        f1 = f1_score(y_va, pred_va, labels=classes, average="macro")
        scores.append(f1); oof_pred[va] = pred_va
        print(f"[PCA+MULTI-EDA-ENS3] Fold {fold} Macro-F1 = {f1:.4f}")

        models.append((scaler, pca, lda, gstats, ens_pack))

    mean_f1 = f1_score(y, oof_pred, labels=classes, average="macro")
    print(f">> PCA+MULTI-EDA-ENS3 OOF Macro-F1 = {mean_f1:.4f} ± {np.std(scores):.4f}")
    print(classification_report(y, oof_pred, labels=classes, digits=4))

    cm = confusion_matrix(y, oof_pred, labels=classes)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title("Confusion Matrix (OOF, PCA+MULTI-EDA-ENS3)")
    plt.show()

    return models, mean_f1

In [ ]:
def predict_hard_pca_multi_eda(test_X, models, target_classes):
    agg = None
    for scaler, pca, lda, gstats, ens_pack in models:
        Z = transform_pca_multi_eda_for_infer(test_X, scaler, pca, lda, gstats, target_classes)
        proba = predict_proba_ens3(ens_pack, Z, target_classes)
        agg = proba if agg is None else (agg + proba)
    return agg / len(models)

In [ ]:
# ============================================
# 4) Stage 1 (Base Multi-EDA) — 3모델 앙상블 CV (bagging)
# ============================================
print("--- 🚀 STAGE 1: 강력한 '일반의' 모델 학습 시작 ---")
test_probas_all = []
skf_stage1 = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, _) in enumerate(skf_stage1.split(X_all, y_all), 1):
    print(f"\n===== STAGE 1 - Fold {fold} =====")
    X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]

    scal, pca, lda, stats, X_tr_final, cls1 = fit_base_multi_eda_s1(X_tr, y_tr, seed=SEED)
    ens_pack = fit_3models_multiclass(
        X_tr_final, y_tr, seed=SEED, use_gpu=USE_GPU, class_order=all_classes
    )
    X_te_final = transform_multi_eda_features_s1(X_test, scal, pca, lda, stats, all_classes)
    proba_fold = predict_proba_ens3(ens_pack, X_te_final, all_classes)
    test_probas_all.append(proba_fold)

proba_full = np.mean(test_probas_all, axis=0)               # (n_test, K)
preds_full_labels = np.array(all_classes)[np.argmax(proba_full, axis=1)]
print("\n--- ✅ STAGE 1: 완료 ---")

# ============================================
# 5) Stage 2 (Hard-class 전용) — 3모델 앙상블 CV
# ============================================
print("\n--- ⚕️ STAGE 2: '전문의' 모델 학습 시작 ---")
filtered = train[train["target"].isin(target_classes)].reset_index(drop=True)
X_hard = filtered.drop(columns=["ID","target"]).values
y_hard = filtered["target"].values

models_hard, hard_oof = run_cv_hard_pca_multi_eda(
    X_hard, y_hard, classes=target_classes, n_splits=NFOLDS, seed=SEED
)
print("--- ✅ STAGE 2: 완료 ---")

# ============================================
# 6) Gating: gap vs hard 비율 분리 집계 + CSV 저장
# ============================================
print("\n--- 🧐 Gating: '전문의'에게 보낼 데이터 선별 ---")

# 안전하게 top2 정렬 후 gap 계산(>=0)
top2 = np.partition(proba_full, -2, axis=1)[:, -2:]
top2.sort(axis=1)
gap = top2[:, 1] - top2[:, 0]

mask_gap  = gap < GAP_THR
mask_hard = np.isin(preds_full_labels, list(HARD_SET))
mask_either = mask_gap | mask_hard
mask_both   = mask_gap & mask_hard
mask_gap_only  = mask_gap  & ~mask_hard
mask_hard_only = mask_hard & ~mask_gap

n_test     = len(test)
n_either   = int(mask_either.sum())
n_both     = int(mask_both.sum())
n_gap_only = int(mask_gap_only.sum())
n_hard_only= int(mask_hard_only.sum())

summary = pd.DataFrame({
    "group": ["gap_only", "hard_only", "both", "TOTAL_gated"],
    "count": [n_gap_only, n_hard_only, n_both, n_either],
})
summary["pct_of_test"]  = summary["count"] / n_test
summary["pct_of_gated"] = summary["count"] / max(n_either, 1)

print("\n[Gating breakdown]")
print(summary.to_string(
    index=False,
    formatters={"pct_of_test":  "{:.3%}".format,
                "pct_of_gated": "{:.3%}".format}
))

# 디버그 CSV 저장
gating_df = pd.DataFrame({
    "ID": test["ID"].values,
    "pred_stage1": preds_full_labels,
    "gap": gap,
    "gap_flag": mask_gap,
    "hard_flag": mask_hard,
    "gated_flag": mask_either,
})
def _reason(g,h):
    if g and h:      return "both"
    if g and not h:  return "gap_only"
    if h and not g:  return "hard_only"
    return "none"
gating_df["reason"] = [_reason(g,h) for g,h in zip(gating_df["gap_flag"], gating_df["hard_flag"])]
gating_df.to_csv(OUT_GATINGCSV, index=False, encoding="utf-8-sig")
print(f"[Saved] gating flags -> {OUT_GATINGCSV}")

# 실제 Stage2 적용
subset_idx = np.where(mask_either)[0]
test_subset = X_test[subset_idx]
print(f"[Gating] 후보 개수: {len(subset_idx)} / {len(test)} (ratio={len(subset_idx)/len(test):.3f})")

proba_hard = predict_hard_pca_multi_eda(test_subset, models_hard, target_classes)
subset_argmax = np.argmax(proba_hard, axis=1)
preds_subset_labels = np.array(target_classes)[subset_argmax]

In [ ]:
# ============================================
# 7) 최종 제출 파일 생성
# ============================================
submission["target"] = preds_full_labels
submission.loc[subset_idx, "target"] = preds_subset_labels
#submission.to_csv(OUT_SUBMIT, index=False, encoding="utf-8-sig")
print(f"\n✅ Saved submission -> {OUT_SUBMIT}")

out_path = "./final_submit_xgb.csv"
submission.to_csv(out_path, index=False, encoding="utf-8-sig")